# Notebook 1: Load and Understand Dataset
This notebook loads the ICU deterioration dataset. Since you are using Google Colab, we will mount your Google Drive first.


In [1]:

# UPDATE THIS PATH to wherever you uploaded your 'preprocessing_pipeline/output' folder in Google Drive
data_dir = 'preprocessing_pipeline/output'


### Load the Flat Dataset
We can explore the flattened CSV format to understand the distribution of features.


In [2]:
import pandas as pd
import os

csv_path = os.path.join(data_dir, 'processed_icu_dataset.csv')
df = pd.read_csv(csv_path)
df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'preprocessing_pipeline/output/processed_icu_dataset.csv'

In [ ]:
print(f'Dataset Shape: {df.shape}')
print('\nData Types and Missing Values:')
df.info()


In [ ]:
print('Summary Statistics:')
df.describe()


### Examine Target Variables (Class Imbalance)


In [ ]:
print('Deterioration within 24 hours:')
print(df['target_24h'].value_counts(normalize=True) * 100)


### Load the 3D LSTM Sequences
The LSTM sequences are stored as `.npy` arrays with shape `(Samples, Sequence_Length, Features)`.


In [ ]:
import numpy as np

X_train = np.load(os.path.join(data_dir, 'X_train.npy'))
y_train = np.load(os.path.join(data_dir, 'y_train.npy'))

print(f'X_train shape: {X_train.shape} (Patients, Hours, Features)')
print(f'y_train shape: {y_train.shape}')


### Train Random Forest and Predict Risk Levels
We will flatten the 3D data, train a model, and categorize patient risk into Low, Moderate, High, and Very High.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import pandas as pd

# Load validation data
X_val = np.load(os.path.join(data_dir, 'X_val.npy'))
y_val = np.load(os.path.join(data_dir, 'y_val.npy'))

# Flatten data for Random Forest
N_train, seq_len, n_features = X_train.shape
N_val = X_val.shape[0]
X_train_flat = X_train.reshape(N_train, seq_len * n_features)
X_val_flat = X_val.reshape(N_val, seq_len * n_features)

# Train the model
print('Training Random Forest...')
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train_flat, y_train)

# Predictions and Probabilities
y_val_prob = rf_model.predict_proba(X_val_flat)[:, 1]

print('ROC-AUC Score:', roc_auc_score(y_val, y_val_prob))


In [ ]:
# Create DataFrame with Risk Levels
results_df = pd.DataFrame({
    'True_Label': y_val,
    'Deterioration_Probability': y_val_prob
})

def categorize_risk(prob):
    if prob < 0.25:
        return 'Low Risk'
    elif prob < 0.50:
        return 'Moderate Risk'
    elif prob < 0.75:
        return 'High Risk'
    else:
        return 'Very High Risk'

results_df['Risk_Level'] = results_df['Deterioration_Probability'].apply(categorize_risk)

print('Sample of Patient Risk Predictions:')
display(results_df[['True_Label', 'Deterioration_Probability', 'Risk_Level']].head(20))

print('\nRisk Level Distribution in Validation Set:')
print(results_df['Risk_Level'].value_counts())
